# Step 1 — Preprocess & Train/Test Split

This notebook cleans your raw CSV and splits it into `train.csv` and `test.csv` **per user**.

**Expected input CSV columns:** `user_id`, `incoming`, `reply`  
**Outputs saved to:** your Google Drive under `MyDrive/llm_project/data/`

Run each cell top to bottom.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install Dependencies

In [ ]:
!pip install -q pandas scikit-learn

## 3. Configuration

**Edit these paths and settings before running.**

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Path to your raw CSV file in Google Drive
RAW_DATA_PATH = 'raw_data.csv'

# Where to save train.csv and test.csv
OUTPUT_DIR = 'data'

# ── Split settings ───────────────────────────────────────────────────────────
# Fraction of each user's data to hold out as the test set
TEST_SIZE = 0.2

# Users with fewer rows than this will be skipped
MIN_SAMPLES = 10

# Random seed for reproducibility
SEED = 42

## 4. Imports

In [ ]:
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

## 5. Load & Inspect Raw Data

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

# Normalise column names
df.columns = df.columns.str.strip().str.lower()

print(f"Raw rows    : {len(df)}")
print(f"Columns     : {list(df.columns)}")
print(f"Users found : {df['user_id'].unique().tolist()}")
print(f"\nRows per user:")
print(df['user_id'].value_counts())
df.head()

Raw rows    : 2637
Columns     : ['rec', 'sent', 'user_id', 'rec_lang', 'sent_lang']
Users found : ['U01', 'U03', 'U14', 'U09', 'U11', 'U12', 'U15']

Rows per user:
user_id
U15    1870
U09     221
U01     184
U12     140
U11     116
U14      68
U03      38
Name: count, dtype: int64


,rec,sent,user_id,rec_lang,sent_lang
0,"[NAME, 8] preciosa es [NAME, 4] que te habla m...",Claro en que,U01,spanish,spanish
1,Tu terminaste la tarea de matematicas,"Cual?, La que mando ayer",U01,spanish,spanish
2,"La de hoy, I dime comiste llagaste bien",No ni he empezado ni me acuerdo q mando estoy ...,U01,spanish,spanish
3,Ohh esta bien cuando la agas me la puedes mand...,Claro que si :),U01,spanish,spanish
4,Gracias me estoy al pegar un Tiro nose por que...,No sigo en.la.busqueda,U01,spanish,spanish


## 6. Clean Data

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    original_len = len(df)

    # Validate required columns
    required = {'user_id', 'rec', 'sent'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing columns: {missing}. Found: {list(df.columns)}")

    # Drop rows with missing values
    df = df.dropna(subset=['user_id', 'rec', 'sent'])

    # Strip whitespace from all string fields
    for col in ['user_id', 'rec', 'sent']:
        df[col] = df[col].astype(str).str.strip()

    # Drop empty strings
    df = df[(df['rec'] != '') & (df['sent'] != '')]

    # Drop exact duplicates per user
    df = df.drop_duplicates(subset=['user_id', 'rec', 'sent'])

    removed = original_len - len(df)
    print(f"Removed {removed} invalid/duplicate rows. Remaining: {len(df)}")
    return df.reset_index(drop=True)

df = clean(df)
df.head()

Removed 0 invalid/duplicate rows. Remaining: 2632


,rec,sent,user_id,rec_lang,sent_lang
0,"[NAME, 8] preciosa es [NAME, 4] que te habla m...",Claro en que,U01,spanish,spanish
1,Tu terminaste la tarea de matematicas,"Cual?, La que mando ayer",U01,spanish,spanish
2,"La de hoy, I dime comiste llagaste bien",No ni he empezado ni me acuerdo q mando estoy ...,U01,spanish,spanish
3,Ohh esta bien cuando la agas me la puedes mand...,Claro que si :),U01,spanish,spanish
4,Gracias me estoy al pegar un Tiro nose por que...,No sigo en.la.busqueda,U01,spanish,spanish


## 7. Per-User Train / Test Split

In [ ]:
def split_per_user(df, test_size, min_samples, seed):
    train_frames, test_frames, skipped = [], [], []

    for user_id, group in df.groupby('user_id'):
        n = len(group)
        if n < min_samples:
            skipped.append((user_id, n))
            continue

        n_test = max(1, int(n * test_size))

        train, test = train_test_split(
            group,
            test_size=n_test,
            random_state=seed,
            shuffle=True
        )
        train_frames.append(train)
        test_frames.append(test)
        print(f"  {str(user_id):20s}  total={n:4d}  train={len(train):4d}  test={len(test):4d}")

    if skipped:
        print(f"\nSkipped {len(skipped)} user(s) with < {min_samples} samples:")
        for uid, n in skipped:
            print(f"  {uid}  (n={n})")

    if not train_frames:
        raise ValueError("No users passed the minimum sample threshold. Lower MIN_SAMPLES or check your data.")

    return (
        pd.concat(train_frames, ignore_index=True),
        pd.concat(test_frames,  ignore_index=True)
    )

print("Per-user split:")
train_df, test_df = split_per_user(df, TEST_SIZE, MIN_SAMPLES, SEED)

Per-user split:
  U01                   total= 184  train= 148  test=  36
  U03                   total=  37  train=  30  test=   7
  U09                   total= 219  train= 176  test=  43
  U11                   total= 115  train=  92  test=  23
  U12                   total= 140  train= 112  test=  28
  U14                   total=  68  train=  55  test=  13
  U15                   total=1869  train=1496  test= 373


## 8. Summary & Save

In [ ]:
print("\n" + "="*50)
print("  Split Summary")
print("="*50)
print(f"  Total train rows : {len(train_df)}")
print(f"  Total test rows  : {len(test_df)}")
print(f"  Users in train   : {train_df['user_id'].nunique()}")
print(f"  Users in test    : {test_df['user_id'].nunique()}")

summary = (
    train_df.groupby('user_id').size().rename('train').to_frame()
    .join(test_df.groupby('user_id').size().rename('test'))
)
summary['total']  = summary['train'] + summary['test']
summary['test_%'] = (summary['test'] / summary['total'] * 100).round(1)
print("\n", summary.to_string())
print("="*50)


  Split Summary
  Total train rows : 2109
  Total test rows  : 523
  Users in train   : 7
  Users in test    : 7

          train  test  total  test_%
user_id                            
U01        148    36    184    19.6
U03         30     7     37    18.9
U09        176    43    219    19.6
U11         92    23    115    20.0
U12        112    28    140    20.0
U14         55    13     68    19.1
U15       1496   373   1869    20.0


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_path = os.path.join(OUTPUT_DIR, 'train.csv')
test_path  = os.path.join(OUTPUT_DIR, 'test.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path,   index=False)

print(f"Saved train.csv → {train_path}")
print(f"Saved test.csv  → {test_path}")
print("\n✅ Preprocessing complete.")